# Calling Package

In [1]:
import pandas as pd
import numpy as np

import sys
import os
import warnings

warnings.filterwarnings("ignore")

# Get the absolute path to the project root (two level up from current directory)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

# Append the project root to sys.path
sys.path.append(project_root)

# Data Ingestion (via CSV)

In [ ]:
df_credit = pd.read_csv("../../data/mock/credit_dummy_data_20250705.csv")

In [ ]:
df_credit['Transaction Datetime'] = pd.to_datetime(df_credit['Transaction Datetime'])

In [ ]:
df_credit['CustomerSex'] = df_credit['CustomerSex'].where(df_credit['CustomerSex'].isin({"0", "1", "2"}), "N")
df_credit['CustomerSex'] = df_credit['CustomerSex'].fillna("N")

# Data Ingestion (via PyODBC)

In [ ]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={SQL Server};SERVER=10.197.51.166;DATABASE=database;UID=user_id;PWD=password"
)

## Prepare Query

In [2]:
from src.utils import read_query_file

# define params
sampling_pct = 0.1
start_date = "2024-12-01"
end_date = "2025-06-01"

# change the directory name with the desired query file name
query = read_query_file("../../sql/join_data/01_credit_card_data.sql")
query = query.format(
    sampling_pct=sampling_pct, start_date=start_date, end_date=end_date
)

In [4]:
print(query)

-- Variable settings:
-- In this query, we sampling down the Clean transactions and deciding the start and end date
DECLARE @sampling_percentage FLOAT = 0.1;
DECLARE @sampling_start_date DATE = '2024-12-01';
DECLARE @sampling_end_date DATE = '2025-06-01';

-- 3 tables that used in the query:
-- Transaction_Summary_Fraud_Hashed
-- C03_Details (channel)
-- C09_Details (channel)
-- Transaction_Summary_Calculations_Fraud_Hashed
WITH tsf_credit AS (
    SELECT
        Transaction_Serial_No
        , Confirmed
    FROM Transaction_Summary_Fraud_Hashed
    WHERE Channel = 'C03'
    OR Channel = 'C09'
)

-- Calling all the channel columns that will be used for Feature Engineering later
, t_union_base AS (
    SELECT
		c03.PANNumber
        , c03.[Transaction Serial No]
		, c03.[Transaction Datetime]
		, c03.[Product Indicator]
		, c03.[Transaction Amount]
		, c03.MCC
		, c03.[Country Code]
		, c03.[Card Acceptor TerminalID] AS 'Card Acceptor Terminal ID'
		, c03.[Card Acceptor ID]
		, c03.[Ter

## Pulling the Data

In [ ]:
df_credit = pd.read_sql(query, conn)
# after pulling the data you can save it via parquet or csv in the desired directory

# Preprocessing

In [ ]:
# this preprocessing is applied for specific columns: MCC, Country Code, Currency Code, POS Mode

def is_convertible_to_float(x):
    try:
        float(x)
        return True
    except (ValueError, TypeError):
        return False

for col in ["MCC", "Country Code", "Currency Code", "POSMode"]:
    # convert column values to string to ensure consistency before applying float conversion
    df_credit[col] = df_credit[col].astype(str)

    if col == "POSMode":
        df_credit[col] = df_credit[col].where(
            # converting the value to float, if it raises ValueError or TypeError (e.g., due to missing or invalid format)
            # then fill the value with "__missing__"
            df_credit[col].apply(is_convertible_to_float), "__missing__"
        )
        df_credit[col] = df_credit[col].replace(
            # replacing string "nan" (case-insensitive) with "__missing__"
            to_replace=r"(?i)^nan$", value="__missing__", regex=True
        )
        # filling any remaining NaN values with "__missing__"
        df_credit[col] = df_credit[col].fillna("__missing__")
    else:
        df_credit[col] = df_credit[col].where(
            # converting the value to float, if it fails then fill with "-999"
            df_credit[col].apply(is_convertible_to_float), "-999"
        )
        df_credit[col] = df_credit[col].replace(
            # replacing string "nan" (case-insensitive) with "-999"
            to_replace=r"(?i)^nan$", value="-999", regex=True
        )
        # filling any remaining NaN values with "-999"
        df_credit[col] = df_credit[col].fillna("-999")


# Feature Engineering

In [ ]:
# Import functionalities
from src.calculation_features import (
    generate_rolling_features,
    label_risk_level_category,
    calculate_time_differences,
    calculate_rolling_txn_hour,
    create_ratio_features
)

In [ ]:
# Import configs
from src.credit_card_config import (
    time_shift_config,
    time_windows,
    freq_config,
    dynamic_high_risk_config,
    duration_since_first_trnx_config,
    unique_count_config,
    monetary_config_1,
    monetary_config_2,
    monetary_config_3,
)
all_monetary_configs = (
    monetary_config_1
    + monetary_config_2
    + monetary_config_3
)

## High Risk Category Features

In [ ]:
df_high_risk_label = label_risk_level_category(
    df=df_credit,
    datetime_col="Transaction Datetime",
    config=dynamic_high_risk_config,
)

## Time (Transaction Hour) Features

In [ ]:
df_txn_hour = calculate_rolling_txn_hour(
    df=df_credit,
    group_col="PANNumber",
    datetime_col="Transaction Datetime",
    windows=time_windows,
)
# AvgTxnHourL15min
# AvgTxnHourL30D

Processing TrnxHour Rolling Avg: 100%|██████████| 5/5 [00:00<00:00, 476.08it/s]


In [10]:
df_txn_hour = create_ratio_features(
    df_txn_hour,
    df_txn_hour.columns,
    "L15min",
    "L30D"
)

## RFM (Recency, Frequency, Monetary) Features

### Recency

#### Transaction Time Difference

In [ ]:
df_time_diff = calculate_time_differences(
    df=df_credit,
    datetime_col="Transaction Datetime",
    groupby_col="PANNumber",
    time_window=time_windows,
    config=time_shift_config,
) # TxnTimeDifference

Calculating rolling averages: 100%|██████████| 2/2 [00:00<00:00, 368.50it/s]


#### Time Duration Since First to Current Transaction

In [ ]:
df_time_firsttxn_to_current = generate_rolling_features(
    df=df_credit,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=duration_since_first_trnx_config,
)
# AvgTimeFirstTxnToCurrentMCCL15min
# AvgTimeFirstTxnToCurrentMCCL30D

Feature Config Progress: 100%|██████████| 2/2 [00:00<00:00, 54.97it/s]


In [13]:
for groupby_col in ["MCC", "Country Code"]:
    df_time_firsttxn_to_current = df_time_firsttxn_to_current.sort_values(by=["PANNumber", "Transaction Datetime"])

    # get first transaction time per target group within each primary group
    df_time_firsttxn_to_current[f"FirstTxnBy{groupby_col.replace(" ","")}"] = df_time_firsttxn_to_current.groupby(["PANNumber", groupby_col])[
        "Transaction Datetime"
    ].transform("first")

    # calculate duration in minutes
    amount_col = f"TimeFirstTxnToCurrent{groupby_col.replace(" ","")}"
    df_time_firsttxn_to_current[amount_col] = (
        df_time_firsttxn_to_current["Transaction Datetime"] - df_time_firsttxn_to_current[f"FirstTxnBy{groupby_col.replace(" ","")}"]
    ).dt.total_seconds() / 60

    df_time_firsttxn_to_current = df_time_firsttxn_to_current.drop(f"FirstTxnBy{groupby_col.replace(" ","")}", axis=1)

In [14]:
# ratio
df_time_firsttxn_to_current = create_ratio_features(
    df_time_firsttxn_to_current,
    df_time_firsttxn_to_current.columns,
    "L15min",
    "L30D"
)

### Frequency

#### Transaction Count

In [ ]:
df_freq = generate_rolling_features(
    df=df_credit,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=freq_config,
)

Feature Config Progress: 100%|██████████| 4/4 [00:00<00:00, 61.98it/s]


In [16]:
# ratio
df_freq = create_ratio_features(
    df_freq,
    df_freq.columns,
    "L15min",
    "L30D"
)

In [ ]:
df_freq.to_parquet("...")
del df_freq

#### Category Unique Count

In [ ]:
df_credit["MCC Num"], uniques = df_credit["MCC"].factorize()
df_credit["PANNumber Num"], uniques = df_credit["PANNumber"].factorize()

df_unique_count = generate_rolling_features(
    df=df_credit,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=unique_count_config,
)

Feature Config Progress: 100%|██████████| 3/3 [00:00<00:00, 48.60it/s]


In [18]:
# ratio
df_unique_count = create_ratio_features(
    df_unique_count,
    df_unique_count.columns,
    "L15min",
    "L30D"
)

### Monetary

In [ ]:
df_monetary = generate_rolling_features(
    df=df_credit,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=all_monetary_configs,
)

Feature Config Progress: 100%|██████████| 9/9 [00:00<00:00, 68.03it/s]


In [20]:
# ratio
df_monetary = create_ratio_features(
    df_monetary,
    df_monetary.columns,
    "L15min",
    "L30D"
)

# Join All Data

In [ ]:
# Define the original columns
og_cols = df_credit.columns.tolist()

# Define the common keys for merging
merge_keys = ["Transaction Serial No", "PANNumber"]

# Extract list of column names from the configs dictionary
time_shift_cols = list(time_shift_config.keys())
freq_cols = [col for col in df_freq.columns if col not in og_cols]
unique_count_cols = [col for col in df_unique_count.columns if col not in og_cols]
monetary_cols = [col for col in df_monetary.columns if col not in og_cols]
time_firsttxn_to_current_cols = [
    col
    for col in df_time_firsttxn_to_current.columns
    if "TimeFirstTxn" in col
]
txn_hour_cols = [
    col
    for col in df_txn_hour.columns
    if "TxnHour" in col
]
high_risk_category_col = [
    col
    for col in df_high_risk_label.columns
    if f"IsTop{dynamic_high_risk_config["top_n"]}HighRisk" in col
]

In [ ]:
from functools import reduce

# Subset dataframes
df_time_diff = df_time_diff[merge_keys + time_shift_cols]
df_freq = df_freq[merge_keys + freq_cols]
df_monetary = df_monetary[merge_keys + monetary_cols]
df_unique_count = df_unique_count[merge_keys + unique_count_cols]
df_time_firsttxn_to_current = df_time_firsttxn_to_current[
    merge_keys + time_firsttxn_to_current_cols
]
df_txn_hour = df_txn_hour[
    merge_keys + txn_hour_cols
]
df_high_risk_label = df_high_risk_label[merge_keys + high_risk_category_col]

# Merge all feature dataframes
dfs_to_merge = [
    df_time_diff,
    df_freq,
    df_monetary,
    df_unique_count,
    df_time_firsttxn_to_current,
    df_txn_hour,
    df_high_risk_label,
]

# Final join
df_credit_join = reduce(
    lambda left, right: pd.merge(left, right, on=merge_keys, how="outer"), dfs_to_merge
)

In [ ]:
# Merge the Feature Engineered columns with Original columns (channel + TSCF)
df_credit_final = df_credit.merge(
    df_credit_join,
    on=merge_keys,
    how="left",
)

# Exploratory Data Analysis

In [10]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_credit_final["log10_TxnTimeDifference"] = np.log10(df_credit_final["TxnTimeDifference"] + 1)

plt.figure(figsize=(8, 5))
sns.boxplot(x="Confirmed", y="log10_TxnTimeDifference", hue="Confirmed", data=df_credit_final)
plt.show()

# Feature Selection

In [ ]:
# do the feature selection
final_column = [
    "column_1",
    "column_2",
]
df_credit_final = df_credit_final[final_column]

# ML Modeling

In [11]:
from src.model_pipeline import ModelPipeline

## Defining Variables in the Pipeline

In [ ]:
# setup pipeline and define model_type
# model_type options: "random_forest", "xgboost", "lightgbm", "lightgbm_balanced"
pipeline = ModelPipeline(model_type="random_forest", random_state=42)

# Fill split_date with the start date of validation/test period
split_date = "2025-05-01"
df_splits = pipeline.split_data_by_date(
    df=df_credit_final, date_column="Transaction Datetime", split_date=split_date
)
# df_splits["df_train"] -> training data
# df_splits["df_test"] -> test/validation data 

In [ ]:
# prepare train data
target_col = "Confirmed"
X, y = pipeline.prepare_data(
    df=df_splits["df_train"],
    target_column=target_col,
    exclude_columns=["Transaction Datetime"],
    encoding_type="OneHot", # Options: "OneHot", "Frequency", "Direct"
    is_apply_log=False, # Is applying log transform in the float columns or not
    is_impute_median=False, # Is using 'median' as the imputation strategy or not
    is_training=True # set True if the dataset is used for training
)

In [ ]:
# split data
X_train, X_test, y_train, y_test = pipeline.split_data(X, y, test_size=0.3)

## Random Over-Sampling

In [ ]:
from src.imbalance_learn import ImbalancedSampler

# OverSampling
ros_sampler = ImbalancedSampler(
    algorithm="RandomOverSampler",
    sampler_type="upsample",
    target_ratio=0.3, # this will give 16.67% minority class proportion vs 83.3% majority
    random_state=1234,
)

X_train_ros, y_train_ros = ros_sampler.fit_resample(X_train, y_train)

## Train Model

In [ ]:
# build and train
pipeline.build_pipeline()
pipeline.train(X_train_ros, y_train_ros, show_training_log=False)

In [ ]:
# returning trained model
base_rf_ros = pipeline.get_model()

In [ ]:
import pickle

# saving model into desired path
filepath = "model/credit/base_rf_ros_38_feats_20250702.pkl"
with open(filepath, "wb") as f:
    pickle.dump(base_rf_ros, f)

In [ ]:
# evaluate test set (30% from the training period)
test_results = pipeline.evaluate(X_test, y_test)

## Back-Testing/Model Validation

In [ ]:
# preprocessed backtest data
X_backtest, y_backtest = pipeline.prepare_data(
    df=df_splits["df_test"],
    target_column=target_col,
    exclude_columns=None,
    # please note that we need to apply the exact variable values in backtest as the one in training
    encoding_type="OneHot", # Options: "OneHot", "Frequency", "Direct"
    is_apply_log=False, # Is applying log transform in the float columns or not
    is_impute_median=False, # Is using 'median' as the imputation strategy or not
    is_training=False # set False since the dataset is used for backtest/validation
)

In [ ]:
# perform model prediction/inference
df_pred = X_backtest[['TransactionAmount']]
df_pred['P_Bad'] = base_rf_ros.predict_proba(X_backtest)[:, 1]
df_pred['Confirmed'] = y_backtest